# CSA0812 – Assignment 3, Q3
## Dictionaries: Farmer Helpline Complaint Analysis

**Student:** Darshan A  
**Register Number:** 1973260011

### Aim
Analyse farmer helpline complaints using dictionaries, sets and `collections.Counter`. Count words while ignoring stop words, group complaints by district and crop, identify the most frequent words, and compare counting methods.

### Prediction
A dictionary and `Counter` should count words in approximately O(n) time, where n is the total number of words. Repeated `list.count()` scans the word list once for each distinct word and therefore takes O(n·d), where d is the number of distinct words. A set provides efficient stop-word membership checks.

## 1. Complete the given code (TODO 1 and TODO 2)

The tie is resolved alphabetically: `price` comes before `received` because both have count 2 and `p` comes before `r`.

In [1]:
calls = [
    "seeds not received seeds delayed",
    "fertiliser price high",
    "seeds price high subsidy not received",
]
STOP = {"not", "high"}

def frequencies(texts, stop):
    counts = {}
    for text in texts:
        for word in text.lower().split():
            if word not in stop:
                counts[word] = counts.get(word, 0) + 1
    return counts

def top(counts, k):
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))[:k]

print(top(frequencies(calls, STOP), 3))

[('seeds', 3), ('price', 2), ('received', 2)]


## 2. District and crop grouping

`setdefault()` creates an empty list the first time a district or crop is seen. Each complaint is then appended to the corresponding list. Punctuation is removed and words are converted to lowercase for consistent analysis.

In [2]:
import re

complaints = [
    {"district": "Thanjavur", "crop": "Rice", "text": "Seeds not received; seeds delayed!"},
    {"district": "Madurai", "crop": "Cotton", "text": "Fertiliser price is high."},
    {"district": "Thanjavur", "crop": "Rice", "text": "Subsidy not received for rice."},
    {"district": "Salem", "crop": "Mango", "text": "Pest control support needed."},
    {"district": "Madurai", "crop": "Cotton", "text": "Cotton seeds are unavailable."},
]

by_district = {}
by_crop = {}

for complaint in complaints:
    by_district.setdefault(complaint["district"], []).append(complaint["text"])
    by_crop.setdefault(complaint["crop"], []).append(complaint["text"])

print("Complaints by district:")
for district, district_complaints in by_district.items():
    print(district, ":", district_complaints)

print("\nComplaints by crop:")
for crop, crop_complaints in by_crop.items():
    print(crop, ":", crop_complaints)

def tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

print("\nTokenisation test:", tokenize("SEEDS, Seeds! PRICE-High."))

Complaints by district:
Thanjavur : ['Seeds not received; seeds delayed!', 'Subsidy not received for rice.']
Madurai : ['Fertiliser price is high.', 'Cotton seeds are unavailable.']
Salem : ['Pest control support needed.']

Complaints by crop:
Rice : ['Seeds not received; seeds delayed!', 'Subsidy not received for rice.']
Cotton : ['Fertiliser price is high.', 'Cotton seeds are unavailable.']
Mango : ['Pest control support needed.']

Tokenisation test: ['seeds', 'seeds', 'price', 'high']


## 3. Word-frequency analysis with stop words

The stop-word set is used to ignore common words. `dict.get()` retrieves the current count (or zero) before adding one. `setdefault()` is demonstrated in the district and crop grouping above.

In [3]:
STOP_WORDS = {
    "a", "an", "and", "are", "for", "from", "high", "in", "is",
    "not", "of", "the", "to", "was", "were"
}

def clean_frequencies(texts, stop):
    counts = {}
    for text in texts:
        for word in tokenize(text):
            if word not in stop:
                counts[word] = counts.get(word, 0) + 1
    return counts

texts = [item["text"] for item in complaints]
word_counts = clean_frequencies(texts, STOP_WORDS)

print("Word frequencies:")
print(word_counts)
print("Top 5 words:", top(word_counts, 5))

Word frequencies:
{'seeds': 3, 'received': 2, 'delayed': 1, 'fertiliser': 1, 'price': 1, 'subsidy': 1, 'rice': 1, 'pest': 1, 'control': 1, 'support': 1, 'needed': 1, 'cotton': 1, 'unavailable': 1}
Top 5 words: [('seeds', 3), ('received', 2), ('control', 1), ('cotton', 1), ('delayed', 1)]


## 4. Experimental investigation: 1,000, 10,000 and 100,000 complaints

The benchmark uses a controlled vocabulary so that the `list.count()` comparison remains practical. Each method receives the same token list. The reported times are measured when this notebook is executed; they are not fixed values.

In [4]:
import time
import statistics
from collections import Counter

VOCAB = [
    "seeds", "fertiliser", "water", "crop", "price",
    "subsidy", "delayed", "received", "farmer", "support",
    "rice", "cotton", "pest", "market", "soil"
]

def generate_complaints(number):
    return [
        f"{VOCAB[i % len(VOCAB)]} {VOCAB[(i + 3) % len(VOCAB)]} "
        f"{VOCAB[(i + 7) % len(VOCAB)]} {VOCAB[(i + 11) % len(VOCAB)]}"
        for i in range(number)
    ]

def dictionary_count(words):
    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1
    return counts

def list_count_method(words):
    distinct = set(words)
    return {word: words.count(word) for word in distinct}

def counter_count(words):
    return Counter(words)

def timed(function, data, repeats=3):
    measurements = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = function(data)
        measurements.append(time.perf_counter() - start)
    return statistics.median(measurements), result

benchmark_results = []

for complaint_total in (1_000, 10_000, 100_000):
    generated = generate_complaints(complaint_total)
    words = [word for complaint in generated for word in complaint.split()]

    dict_time, dict_result = timed(dictionary_count, words)
    list_time, list_result = timed(list_count_method, words)
    counter_time, counter_result = timed(counter_count, words)

    assert dict_result == list_result == dict(counter_result)

    benchmark_results.append({
        "Complaints": complaint_total,
        "Total words": len(words),
        "Distinct words": len(set(words)),
        "Dictionary (s)": dict_time,
        "list.count() (s)": list_time,
        "Counter (s)": counter_time,
        "Results match": True
    })

print("Benchmark results (median of 3 runs):")
for row in benchmark_results:
    print(row)

Benchmark results (median of 3 runs):
{'Complaints': 1000, 'Total words': 4000, 'Distinct words': 15, 'Dictionary (s)': 0.0007666550000067218, 'list.count() (s)': 0.0018888089998654323, 'Counter (s)': 0.0002883440001824056, 'Results match': True}
{'Complaints': 10000, 'Total words': 40000, 'Distinct words': 15, 'Dictionary (s)': 0.00442033299987088, 'list.count() (s)': 0.011102688999926613, 'Counter (s)': 0.0033452299999225943, 'Results match': True}
{'Complaints': 100000, 'Total words': 400000, 'Distinct words': 15, 'Dictionary (s)': 0.04628831799982436, 'list.count() (s)': 0.12123962500004382, 'Counter (s)': 0.04530433800005085, 'Results match': True}


## 5. Edge-case tests: empty text, case and punctuation

In [5]:
edge_tests = [
    [],
    [""],
    ["SEEDS seeds Seeds"],
    ["Seeds, seeds! SEEDS?"],
    ["not NOT Not"],
]

for sample in edge_tests:
    print("Input:", sample)
    print("Counts:", clean_frequencies(sample, STOP_WORDS))
    print()

# The first two inputs produce empty dictionaries.
assert clean_frequencies([], STOP_WORDS) == {}
assert clean_frequencies([""], STOP_WORDS) == {}

# Case and punctuation are normalised.
assert clean_frequencies(["SEEDS, seeds! SEEDS?"], STOP_WORDS) == {"seeds": 3}
print("All edge-case assertions passed.")

Input: []
Counts: {}

Input: ['']
Counts: {}

Input: ['SEEDS seeds Seeds']
Counts: {'seeds': 3}

Input: ['Seeds, seeds! SEEDS?']
Counts: {'seeds': 3}

Input: ['not NOT Not']
Counts: {}

All edge-case assertions passed.


## 6. Design and analysis

- **How dictionary lookup works:** Python computes a hash for the key and uses it to locate a candidate slot in the hash table. Equality checks resolve collisions. With a well-distributed hash table, lookup is O(1) on average, so expected lookup cost does not grow linearly with the number of stored keys. Worst-case collisions can make lookup slower.
- **Dictionary vs `list.count()`:** a dictionary loop updates each word count once, giving O(n) expected time. Calling `list.count()` for each of d distinct words scans all n words each time, giving O(n·d) time.
- **Dictionary order:** Python 3.7 and later guarantee that dictionaries preserve insertion order. Updating an existing key does not move it; deleting and reinserting a key places it at the end.
- **Views:** `keys()`, `values()` and `items()` return dynamic view objects, not independent lists. They reflect later changes to the dictionary. `items()` provides key-value pairs.
- **Space:** dictionary and `Counter` counting use O(d) additional space for d distinct words. The `list.count()` approach also needs a distinct-key collection/result of O(d), while the input word list occupies O(n) space.
- **Engineering decision:** for 100,000 complaints per day, deploy the dictionary counting approach or `Counter`. Both provide expected O(n) counting and are substantially more scalable than repeated `list.count()`. `Counter` is concise and purpose-built; a dictionary is suitable when custom filtering or update logic is required.

## 7. Conclusion

Dictionaries efficiently count words and group complaints by district and crop. A set makes stop-word filtering straightforward. Sorting by negative frequency and then alphabetically produces the required top-k order. The benchmark compares dictionary counting, repeated `list.count()` and `Counter` on the same generated data, while edge-case tests check empty input, case normalisation and punctuation handling.

**Screenshot evidence:** Capture the completed-code output, district/crop grouping output, benchmark results table/output, and edge-case test output after running the notebook in Google Colab.